In [ ]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

REQUIRE_CUDA = True  # set False for CPU-only notebooks

print(f"CPU threads set to: {CPU_THREADS}")

try:
    import torch
except Exception as e:
    torch = None
    if REQUIRE_CUDA:
        raise RuntimeError("CUDA required but torch is not available.") from e

if torch is not None:
    torch.set_num_threads(CPU_THREADS)
    torch.set_num_interop_threads(min(4, CPU_THREADS))
    if REQUIRE_CUDA and not torch.cuda.is_available():
        raise RuntimeError("CUDA required but not available.")
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        print("CUDA device:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available; running on CPU.")



# LLaVA-1.5 7B zero-shot baseline — Kvasir-VQA x1

Single-GPU zero-shot evaluation on the x1 validation + test splits. Reuses the x1 answer normalization/top-K mapping so we can compare with BLIP/BLIP-2 style baselines. No training.


In [1]:

# !pip install sentencepiece

import os
import json
import random
import re
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from sklearn.metrics import classification_report, f1_score
from transformers import AutoProcessor, LlavaForConditionalGeneration

try:
    from transformers import BitsAndBytesConfig
except Exception:
    BitsAndBytesConfig = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


2026-01-27 07:45:55.782482: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-27 07:45:55.782514: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-27 07:45:55.783534: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-27 07:45:55.789971: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-27 07:45:57.022921: W tensorflow/compiler/tf2

In [2]:

# Paths & run config

def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"

RUN_NAME = "03_vlm_modern_baseline_zeroshot"
MODEL_NAME = os.environ.get("VLM_MODEL_NAME", "llava-hf/llava-1.5-7b-hf")
SPLITS = ["validation", "test"]
SAMPLE_N: Optional[int] = None  # set an int to subsample each split for smoke tests
TOP_K = 200  # follow x1 top-K convention

BATCH_SIZE = 2  # adjust if OOM
MAX_NEW_TOKENS = 16
USE_4BIT = True   # preferred for single 24/32GB GPUs
USE_8BIT = False  # fallback if 4-bit unavailable

OUT_DIR = DATA_ROOT / "2_modeling" / RUN_NAME / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Output dir:", OUT_DIR)
print("Model:", MODEL_NAME)


Data root: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Output dir: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/03_vlm_modern_baseline_zeroshot/out
Model: llava-hf/llava-1.5-7b-hf


In [3]:

# Load metadata & build top-K answers from train split

def basic_norm(text: str) -> str:
    t = str(text).lower().strip()
    t = re.sub(r"[^a-z0-9\s\-]", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

meta_all = pd.read_csv(META_CSV)
images_base = DATA_ROOT / "0_dataset_prep"
meta_all["image_path"] = meta_all["image_path"].apply(
    lambda p: str((images_base / p).resolve()) if not Path(p).is_absolute() else p
)

# Basic cleaning
meta_all = meta_all.dropna(subset=["question", "answer", "image_path"]).reset_index(drop=True)
meta_all = meta_all[meta_all["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)

meta_all["question_norm"] = meta_all["question"].astype(str).str.strip()
meta_all["answer_norm"] = meta_all["answer"].apply(basic_norm)

train_df = meta_all[meta_all["split"] == "train"].reset_index(drop=True)
answer_counts = train_df["answer_norm"].value_counts()
TOP_ANSWERS = answer_counts.head(TOP_K).index.tolist()
print("Top-K answers (train):", len(TOP_ANSWERS))

meta_all = meta_all[meta_all["split"].isin(SPLITS)].reset_index(drop=True)
print({s: len(meta_all[meta_all['split'] == s]) for s in SPLITS})
meta_all.head()


Top-K answers (train): 200
{'validation': 5928, 'test': 5947}


,split,img_id,image_path,question,answer,question_type,answer_type,orig_height,orig_width,question_norm,answer_norm
0,validation,cla820gl1s3pv071u6kb331te,/home/aristotle/Desktop/rag-vqa-medical/Protot...,Where in the image is the instrument?,none,NaN,NaN,576,720,Where in the image is the instrument?,none
1,validation,cla820gl1s3pv071u6kb331te,/home/aristotle/Desktop/rag-vqa-medical/Protot...,Is this finding easy to detect?,yes,NaN,NaN,576,720,Is this finding easy to detect?,yes
2,validation,cla820gl1s3pv071u6kb331te,/home/aristotle/Desktop/rag-vqa-medical/Protot...,Where in the image is the anatomical landmark?,none,NaN,NaN,576,720,Where in the image is the anatomical landmark?,none
3,validation,cla820gl1s3pv071u6kb331te,/home/aristotle/Desktop/rag-vqa-medical/Protot...,Where in the image is the abnormality?,center; center-left; center-right; lower-cente...,NaN,NaN,576,720,Where in the image is the abnormality?,center center-left center-right lower-center l...
4,validation,cla820gl1s3pv071u6kb331te,/home/aristotle/Desktop/rag-vqa-medical/Protot...,What is the size of the polyp?,none,NaN,NaN,576,720,What is the size of the polyp?,none



### Normalization + mapping to top-K
- Lowercase and strip punctuation/extra spaces.
- Map predictions to the top-K answer list; if no good match, map to `other`.
- Ground-truth answers are also mapped to top-K (else `other`) so metrics stay comparable.


In [4]:

import difflib

_top_set = set(TOP_ANSWERS)


def normalize_answer(text: str) -> str:
    t = str(text).lower().strip()
    t = re.sub(r"[^a-z0-9\s\-]", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


def map_to_topk(ans_norm: str) -> str:
    if ans_norm in _top_set:
        return ans_norm
    match = difflib.get_close_matches(ans_norm, TOP_ANSWERS, n=1, cutoff=0.75)
    if match:
        return match[0]
    return "other"



### Load LLaVA-1.5 7B (quantized if possible)
Prefers 4-bit via bitsandbytes; falls back to 8-bit or full precision depending on availability.


In [5]:

quant_config = None
if torch.cuda.is_available() and BitsAndBytesConfig is not None:
    if USE_4BIT:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )
        print("Using 4-bit quantization (bitsandbytes).")
    elif USE_8BIT:
        quant_config = BitsAndBytesConfig(load_in_8bit=True)
        print("Using 8-bit quantization (bitsandbytes).")

processor = AutoProcessor.from_pretrained(MODEL_NAME, use_fast=False)
load_kwargs = {
    "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
    "low_cpu_mem_usage": True,
}
if torch.cuda.is_available():
    load_kwargs["device_map"] = "auto"
if quant_config is not None:
    load_kwargs["quantization_config"] = quant_config

model = LlavaForConditionalGeneration.from_pretrained(MODEL_NAME, **load_kwargs)
model.eval()

if hasattr(model, "hf_device_map"):
    device_vals = set(str(d) for d in model.hf_device_map.values())
    primary = next((d for d in device_vals if "cuda" in d), None)
    if primary is None:
        primary = next((d for d in device_vals if d not in {"cpu", "disk"}), "cpu")
    PRIMARY_DEVICE = torch.device(primary)
else:
    PRIMARY_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Primary device for inputs:", PRIMARY_DEVICE)


Using 4-bit quantization (bitsandbytes).


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Primary device for inputs: cpu



### Prompting + generation helpers
Uses the model chat template when available; otherwise falls back to `USER: <image> Question: ... ASSISTANT:` format.


In [6]:


def format_prompt(question: str) -> str:
    q = question.strip()
    if hasattr(processor, "apply_chat_template"):
        conversation = [
            {"role": "system", "content": "You are a concise medical VQA assistant."},
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": q}]},
        ]
        return processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
    return f"USER: <image>\nQuestion: {q}\nASSISTANT:"


def postprocess_output(text: str) -> str:
    cleaned = text
    if "ASSISTANT:" in cleaned:
        cleaned = cleaned.split("ASSISTANT:")[-1]
    return cleaned.strip()


def chunk_indices(n: int, batch_size: int):
    for start in range(0, n, batch_size):
        yield start, min(start + batch_size, n)


def generate_for_split(df_split: pd.DataFrame, split_name: str) -> pd.DataFrame:
    df_split = df_split.reset_index(drop=True).copy()
    preds = []
    for start, end in tqdm(list(chunk_indices(len(df_split), BATCH_SIZE)), desc=f"{split_name} gens"):
        batch = df_split.iloc[start:end]
        images = [Image.open(p).convert("RGB") for p in batch["image_path"]]
        prompts = [format_prompt(q) for q in batch["question"]]
        inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
        inputs = {k: v.to(PRIMARY_DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            out_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        decoded = processor.batch_decode(out_ids, skip_special_tokens=True)
        preds.extend(postprocess_output(t) for t in decoded)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    df_split["pred_raw"] = preds
    df_split["pred_norm"] = df_split["pred_raw"].apply(normalize_answer)
    df_split["pred_mapped"] = df_split["pred_norm"].apply(map_to_topk)
    df_split["gt_norm"] = df_split["answer_norm"].apply(normalize_answer)
    df_split["gt_mapped"] = df_split["gt_norm"].apply(map_to_topk)
    df_split["is_oov"] = df_split["pred_mapped"] == "other"
    return df_split


def compute_metrics(df: pd.DataFrame, split_name: str):
    labels_eval = TOP_ANSWERS + ["other"]
    metrics = {
        "split": split_name,
        "n": int(len(df)),
        "mapped_acc": float((df["pred_mapped"] == df["gt_mapped"]).mean()),
        "macro_f1": float(f1_score(df["gt_mapped"], df["pred_mapped"], labels=labels_eval, average="macro", zero_division=0)),
        "oov_rate": float(df["is_oov"].mean()),
    }
    report = classification_report(
        df["gt_mapped"], df["pred_mapped"], labels=labels_eval, output_dict=True, zero_division=0
    )
    per_class = pd.DataFrame(report).T
    return metrics, per_class



### Run zero-shot inference on validation + test
Saves per-sample predictions, per-class metrics, metrics summary, and a 20-row qualitative table per split.


In [8]:

all_metrics: List[Dict] = []
per_class_store: Dict[str, pd.DataFrame] = {}
qual_store: Dict[str, pd.DataFrame] = {}

for split in SPLITS:
    df_split = meta_all[meta_all["split"] == split].copy()
    if SAMPLE_N is not None:
        df_split = df_split.sample(min(SAMPLE_N, len(df_split)), random_state=SEED).reset_index(drop=True)
    print(f"=== {split.upper()} ({len(df_split)} samples) ===")
    df_pred = generate_for_split(df_split, split)
    metrics, per_class = compute_metrics(df_pred, split)
    all_metrics.append(metrics)
    per_class_store[split] = per_class

    df_pred.to_csv(OUT_DIR / f"predictions_{split}.csv", index=False)
    per_class.to_csv(OUT_DIR / f"per_class_{split}.csv")

    qual = df_pred.sample(n=min(20, len(df_pred)), random_state=SEED)[
        ["img_id", "question", "pred_raw", "pred_mapped", "gt_mapped", "answer", "is_oov"]
    ]
    qual.to_csv(OUT_DIR / f"qualitative_{split}.csv", index=False)
    qual_store[split] = qual

metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(OUT_DIR / "metrics_summary.csv", index=False)
with open(OUT_DIR / "metrics_summary.json", "w") as f:
    json.dump(all_metrics, f, indent=2)

metrics_df


=== VALIDATION (5928 samples) ===


validation gens:   0%|          | 0/2964 [00:00<?, ?it/s]

KeyboardInterrupt: 


### Quick qualitative peek (test split)


In [ ]:
qual_store.get("test", pd.DataFrame()).head(20)